# B3 · Many correlated features: OLS, Ridge, regression on the components and PLS

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Compare four ways of handling 40 correlated features (generated from 5 latent factors) with only 100 training examples: linear regression, Ridge, **principal component regression** (PCR: PCA + linear regression) with `k` chosen in cross-validation, and **PLS**
(components chosen by looking at `y` too).

## Dataset

`generate()` (given): 200 examples, 40 features built as combinations of 5 latent factors plus noise; `y` depends on 3 of the 5 factors (plus noise). 50/50 split: `X_tr, X_te, y_tr, y_te`.

## Educational purpose

Recognize the classic case for which PCR exists (highly correlated features, few useful directions) and see its limit: the components are chosen without looking at `y`. PLS is the supervised alternative.

## Task

1. Compute `test_r2` (dictionary) with the test R² of `OLS` (`LinearRegression`), `Ridge` (`RidgeCV(alphas=np.logspace(-2, 4, 20))`), `PCR` and `PLS` (see below).
2. **PCR:** for `k` in `range(1, 21)` compute `cv_pcr` (mean R² in `KFold(5, shuffle=True, random_state=0)` on the training set) of the pipeline `PCA(k)` → `LinearRegression`. Save `pcr_k`: the **smallest** `k` with `cv_pcr` within 0.01 of the maximum.
3. **PLS:** same procedure with `PLSRegression(n_components=k)` for `k` in `range(1, 11)` (`cv_pls`) and `pls_n` (smallest `k` within 0.01 of the maximum).
4. Save `pls_has_fewer_components` (True if `pls_n < pcr_k`) and fill `test_r2` with the final models trained with `pcr_k` and `pls_n`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline


def generate():
    rng = np.random.default_rng(0)
    n = 200
    L = rng.normal(size=(n, 5))                                       # 5 latent factors
    X = L @ rng.normal(size=(5, 40)) + rng.normal(0, 1.0, (n, 40))    # 40 correlated features
    y = L @ np.array([3, -2, 1, 0.5, 0]) + rng.normal(0, 1.5, n)
    return train_test_split(X, y, test_size=0.5, random_state=0)


X_tr, X_te, y_tr, y_te = generate()
cv = KFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
cv_pcr = pd.Series(dtype=float)
pcr_k = None

In [ ]:
cv_pls = pd.Series(dtype=float)
pls_n = None
pls_has_fewer_components = None

In [ ]:
test_r2 = {}
test_r2

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`cross_val_score(make_pipeline(PCA(k), LinearRegression()), X_tr, y_tr, cv=cv, scoring="r2").mean()`. PLS: `cross_val_score(PLSRegression(k), X_tr, y_tr, cv=cv, scoring="r2")`. Final models: `.fit(X_tr, y_tr).score(X_te, y_te)`. Rule: `min(k for k in series.index if series[k] >= series.max() - 0.01)`.'):
    check(list(cv_pcr.index) == list(range(1, 21)) and cv_pcr.idxmax() >= 4 and cv_pcr.max() > 0.7, "PCR curve over components 1-20: maximum above 0.7 with at least 4 components", "PCA(k) + LinearRegression")
    check(cv_pcr[1] < 0.3 and 4 <= pcr_k <= 10, "with a single component the R2 is low (below 0.3); the chosen k is between 4 and 10", "smallest k within 0.01 of the maximum")
    check(list(cv_pls.index) == list(range(1, 11)) and pls_n <= 4 and pls_has_fewer_components is True, "PLS: at most 4 components, fewer than PCR", "PLS chooses the directions by looking at y")
    check(set(test_r2) == {"OLS", "Ridge", "PCR", "PLS"} and test_r2["OLS"] < 0.7 and test_r2["Ridge"] > 0.78 and test_r2["PCR"] > 0.78 and test_r2["PLS"] > 0.75, "test R2: OLS below 0.7; Ridge and PCR above 0.78; PLS above 0.75", "final models trained on the training set")
    check(test_r2["PCR"] > test_r2["OLS"] + 0.1, "PCR beats OLS by over 0.1 of R2", "many correlated features and few examples: OLS overfits")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_regression_on_components_and_pls.ipynb`. Open it after you pass the check, or after you have really tried.